# invest_panel_weo：合并与数据质量核验

## tl;dr

- 输出为 1,972 行、24 列、68 个国家/地区的 1995–2023 平衡面板。
- `iso3 + year` 无重复，合并没有改变基础面板行数。
- 新增 WEO 列覆盖率：capitaGDP 99.80%，revenue 96.75%，debt 94.62%。
- 派生列 interest_revenue 覆盖率为 92.44%，公式最大绝对误差为 1.42e-14。
- WEO 百分数保持原始百分比点单位，没有乘以 100。


## Context & Methods

本 notebook 是 CSV 与说明文档的审计附件。它重新读取基础面板、输出面板和 WEO 6 个目标系列，检查字段保留、唯一键、左连接行数、WEO 数值一致性、派生公式、缺失率和描述统计。

### Key Assumptions

- `iso3 + year` 是目标面板唯一键。
- WEO `COUNTRY.ID` 与基础面板 `iso3` 可直接匹配。
- 分析期限定为 1995–2023。


## Data

### 1. Load inputs and output

In [1]:
from pathlib import Path
import importlib.util
import sys
import pandas as pd

sys.dont_write_bytecode = True
OUTPUT_DIR = Path.cwd()
PROJECT_ROOT = OUTPUT_DIR.parent
BASE_CSV = PROJECT_ROOT / "cleaned_imf_like_panel_1995_2023.csv"
OUTPUT_CSV = OUTPUT_DIR / "invest_panel_weo.csv"

base = pd.read_csv(BASE_CSV)
panel = pd.read_csv(OUTPUT_CSV)

spec = importlib.util.spec_from_file_location("builder", OUTPUT_DIR / "build_invest_panel_weo.py")
builder = importlib.util.module_from_spec(spec)
spec.loader.exec_module(builder)
_, weo_numeric, weo_metadata, _ = builder.read_weo_values()

base.shape, panel.shape, weo_metadata.groupby("code")["iso3"].nunique().to_dict()


((1972, 17),
 (1972, 24),
 {'GGR': 197,
  'GGR_NGDP': 197,
  'GGXCNL_NGDP': 197,
  'GGXWDG': 197,
  'NGDP': 197,
  'NGDPRPPPPC': 197})

## Results

### 2. Validate grain and source-field preservation

In [2]:
expected_columns = ["PrimaryBalance_gdp" if c == "OB_gdp" else c for c in base.columns] + list(builder.WEO_FIELDS.values()) + builder.DERIVED_FIELDS

renamed_base = base.rename(columns={"OB_gdp": "PrimaryBalance_gdp"})
preserved = renamed_base.equals(panel[renamed_base.columns])
checks = pd.Series({
    "output_rows": len(panel),
    "output_columns": panel.shape[1],
    "column_order_matches": panel.columns.tolist() == expected_columns,
    "base_values_preserved": preserved,
    "duplicate_iso3_year_keys": int(panel.duplicated(["iso3", "year"]).sum()),
    "exact_duplicate_rows": int(panel.duplicated().sum()),
    "countries": int(panel["iso3"].nunique()),
    "year_min": int(panel["year"].min()),
    "year_max": int(panel["year"].max()),
})
checks


output_rows                 1972
output_columns                24
column_order_matches        True
base_values_preserved       True
duplicate_iso3_year_keys       0
exact_duplicate_rows           0
countries                     68
year_min                    1995
year_max                    2023
dtype: object

### 3. Reconcile appended values to WEO

In [3]:
weo_rows = [
    {"iso3": iso3, "year": year, "code": code, "source_value": value}
    for (iso3, year, code), value in weo_numeric.items()
]
weo_long = pd.DataFrame(weo_rows)
weo_wide = weo_long.pivot(index=["iso3", "year"], columns="code", values="source_value").reset_index()
weo_wide = weo_wide.rename(columns=builder.WEO_FIELDS)
reconciled = panel.merge(weo_wide, on=["iso3", "year"], how="left", suffixes=("_out", "_src"), validate="one_to_one")

reconciliation = {}
for column in builder.WEO_FIELDS.values():
    out = reconciled[f"{column}_out"]
    src = reconciled[f"{column}_src"]
    both = out.notna() & src.notna()
    reconciliation[column] = {
        "output_non_missing": int(out.notna().sum()),
        "source_non_missing": int(src.notna().sum()),
        "max_absolute_difference": float((out[both] - src[both]).abs().max()) if both.any() else None,
        "missingness_matches": bool(out.isna().equals(src.isna())),
    }
display(pd.DataFrame(reconciliation).T)

expected_interest = (panel["PrimaryBalance_gdp"] - panel["OverallBalance_gdp"]) / panel["Revenue_gdp"] * 100
expected_interest_mask = panel[["PrimaryBalance_gdp", "OverallBalance_gdp", "Revenue_gdp"]].notna().all(axis=1) & panel["Revenue_gdp"].ne(0)
interest_check = pd.Series({
    "output_non_missing": int(panel["interest_revenue"].notna().sum()),
    "expected_non_missing": int(expected_interest_mask.sum()),
    "missingness_matches": bool(panel["interest_revenue"].notna().equals(expected_interest_mask)),
    "max_absolute_difference": float((panel.loc[expected_interest_mask, "interest_revenue"] - expected_interest[expected_interest_mask]).abs().max()),
    "zero_revenue_gdp_rows": int(panel["Revenue_gdp"].eq(0).sum()),
})
interest_check


,output_non_missing,source_non_missing,max_absolute_difference,missingness_matches
Revenue_gdp,1908,1908,0.0,True
CurrentGDP,1970,1970,0.0,True
OverallBalance_gdp,1901,1901,0.0,True
revenue,1908,1908,0.0,True
debt,1866,1866,0.0,True
capitaGDP,1968,1968,0.0,True


output_non_missing         1823
expected_non_missing       1823
missingness_matches        True
max_absolute_difference     0.0
zero_revenue_gdp_rows         0
dtype: object

### 4. Review completeness

In [4]:
coverage = pd.DataFrame({
    "non_missing": panel.notna().sum(),
    "missing": panel.isna().sum(),
    "coverage_pct": panel.notna().mean().mul(100),
})
coverage


,non_missing,missing,coverage_pct
country_name,1972,0,100.000000
iso3,1972,0,100.000000
year,1972,0,100.000000
bond_spreads,1447,525,73.377282
bond_10y,1447,525,73.377282
vulnerability100,1914,58,97.058824
readiness100,1914,58,97.058824
lnrgdp,1965,7,99.645030
growth,1961,11,99.442191
inflation_cpi,1965,7,99.645030


### 5. Review numeric distributions

In [5]:
panel.describe(percentiles=[0.25, 0.5, 0.75]).T


,count,mean,std,min,25%,50%,75%,max
year,1972.0,2009.000000,8.368722e+00,1995.000000,2002.000000,2009.000000,2016.000000,2.023000e+03
bond_spreads,1447.0,2.291850,4.517004e+00,-3.405732,-0.493696,0.606927,3.814549,3.430871e+01
bond_10y,1447.0,5.530201,4.490886e+00,-0.505896,2.696218,4.503737,7.016413,3.520283e+01
vulnerability100,1914.0,38.177320,7.812113e+00,25.103007,31.728209,36.622276,43.387361,5.807999e+01
readiness100,1914.0,49.862456,1.422730e+01,17.934155,36.907749,49.430339,61.385029,8.072020e+01
lnrgdp,1965.0,7.845485,2.922433e+00,1.894617,5.789792,7.649359,9.565450,1.632523e+01
growth,1961.0,3.413316,3.575546e+00,-16.040000,1.700000,3.496000,5.513000,2.462400e+01
inflation_cpi,1965.0,5.650752,1.062028e+01,-3.967000,1.615000,3.143000,6.548000,1.973000e+02
debt_gdp,1866.0,56.833995,3.443220e+01,0.052000,34.521500,50.578500,71.171000,2.609640e+02
PrimaryBalance_gdp,1823.0,-0.504317,3.378360e+00,-29.952000,-2.360000,-0.519000,1.364000,2.344900e+01


## Takeaways

- 面板键和行数检查通过，基础字段在改名后逐值保持一致。
- 6 个 WEO 字段与源值及缺失位置一致，未发生单位缩放。
- `capitaGDP` 为不变价、购买力平价口径的实际人均 GDP，可用于跨国水平比较，但不应解释为当期市场汇率美元收入。
- `interest_revenue` 严格按 `((PrimaryBalance_gdp - OverallBalance_gdp) / Revenue_gdp) * 100` 计算，单位为百分数。
- 财政收入和总体余额缺失主要发生在样本早期；建模时应记录最终可用样本。
- CurrentGDP、revenue 和 debt 为十亿本币，适合国别内变化分析，不适合未经汇率或 PPP 转换的跨国水平比较。
